# Memory bandwidth and extra checks for the paper replication (Kaggle, T4)

Independent checks of the analysis: STREAM-like bandwidth (not matvec), FC with B vectors, conv split
into im2col and GEMM. 3 runs of `bench_mem.py`, about 10 minutes in total.

**Before running**
1. Settings → Accelerator → **GPU T4 x2**.
2. Run All. Predictions were written before any run (`PREDICTIONS.md` in the repository).
3. Download `results_bandwidth.zip` from the Output panel, then stop the session.

The code cells below are generated from the project's `code/` folder by `make_bandwidth_notebook.py`.

## 1 · Machine check

In [ ]:
!nvidia-smi
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Core|Socket|L2|L3'
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2, then restart the session."
print("torch CPU threads (max):", torch.get_num_threads())
!mkdir -p /kaggle/working/code

## 2 · Benchmark code (written to `/kaggle/working/code`)

In [ ]:
%%writefile /kaggle/working/code/common.py
"""Shared helpers for the benchmarks: devices, threads, timing, machine info, CSV output."""
import csv
import json
import os
import platform
import random
import statistics
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import torch

ROOT = Path(__file__).resolve().parent.parent

# torch picks one intra-op thread per physical core by default. Capture it before
# any benchmark calls torch.set_num_threads(), so "max" always means this value.
DEFAULT_THREADS = torch.get_num_threads()

CSV_FIELDS = [
    "exp", "impl", "device", "threads", "size",
    "mean_ms", "std_ms", "min_ms", "median_ms", "issue_ms", "warmup", "repeats",
    "gflops", "gbps", "order", "t_start", "t_end", "machine", "timestamp",
]


def parse_ints(text):
    return [int(v) for v in text.split(",") if v.strip()]


def resolve_threads(spec):
    """'1,max' -> [1, DEFAULT_THREADS] (duplicates removed, order kept)."""
    values = []
    for item in spec.split(","):
        item = item.strip()
        if item == "max":
            values.append(DEFAULT_THREADS)
        elif item:
            values.append(int(item))
    return list(dict.fromkeys(values))


def get_device(name):
    if name == "cuda":
        if not torch.cuda.is_available():
            raise SystemExit("CUDA is not available on this machine.")
        # The paper's GPU (Kepler) only had plain FP32. Turn off TF32 shortcuts so
        # newer GPUs compute in true FP32 too.
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = True
    elif name != "cpu":
        raise SystemExit(f"Unknown device '{name}' (use cpu or cuda).")
    return torch.device(name)


def available_devices():
    return ["cpu", "cuda"] if torch.cuda.is_available() else ["cpu"]


def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def bench(fn, device, warmup=3, repeat=10):
    """Run fn `warmup` times untimed, then `repeat` timed runs (wall clock, ms).

    GPU work is asynchronous, so the device is synchronized before starting and
    after finishing each timed run: the time covers the full kernel execution.

    issue_ms is the part until fn() returns, i.e. before the final synchronize. On a GPU
    that is the host's launch work (as long as the launches fit in the CUDA queue), so
    issue_ms ~ mean_ms means the run is bound by the CPU launching kernels, and
    issue_ms << mean_ms means it is bound by the GPU executing them.
    """
    for _ in range(warmup):
        fn()
    sync(device)
    times, issue = [], []
    t_start = time.time()
    for _ in range(repeat):
        sync(device)
        start = time.perf_counter()
        fn()
        issue.append((time.perf_counter() - start) * 1000.0)
        sync(device)
        times.append((time.perf_counter() - start) * 1000.0)
    return {
        "issue_ms": statistics.fmean(issue),
        "t_start": t_start,
        "t_end": time.time(),
        "mean_ms": statistics.fmean(times),
        "std_ms": statistics.stdev(times) if len(times) > 1 else 0.0,
        "min_ms": min(times),
        "median_ms": statistics.median(times),
    }


def ordered(items, seed):
    """Measurement order. Seed 0 keeps it ascending; any other seed shuffles reproducibly.

    With an ascending order "larger batch" and "later in time" cannot be told apart, e.g.
    a GPU raising its clocks during the sweep would look like a batch-size effect.
    """
    items = list(items)
    if seed:
        random.Random(seed).shuffle(items)
    return items


def gpu_warmup(device, seconds):
    """Keep the GPU busy so it reaches its sustained clocks before anything is timed."""
    if device.type != "cuda" or seconds <= 0:
        return
    a = torch.randn(4096, 4096, device=device)
    b = torch.empty_like(a)
    end = time.perf_counter() + seconds
    while time.perf_counter() < end:
        torch.mm(a, a, out=b)
        torch.cuda.synchronize(device)
    del a, b


class GpuClockLog:
    """Log GPU 0 clocks/power/temperature (gpu_sampler.py, every 20 ms) while the block runs."""

    def __init__(self, path, enabled, log=print):
        self.path, self.enabled, self.log, self.proc = Path(path), enabled, log, None

    def __enter__(self):
        if self.enabled:
            self.err = self.path.with_suffix(".err")
            self.err_fh = self.err.open("w", encoding="utf-8")
            self.proc = subprocess.Popen(
                [sys.executable, str(Path(__file__).with_name("gpu_sampler.py")), str(self.path), "20"],
                stderr=self.err_fh)
            time.sleep(1)
        return self

    def __exit__(self, *exc):
        if self.proc is None:
            return
        self.proc.terminate()
        self.proc.wait(timeout=10)
        self.err_fh.close()
        samples = max(len(self.path.read_text().splitlines()) - 1, 0) if self.path.exists() else 0
        error = self.err.read_text(encoding="utf-8").strip().splitlines()
        self.log(f"gpu clock log: {samples} samples" + (f" | ERROR: {error[-1]}" if error else ""))


_order = 0


def make_row(exp, impl, device, threads, size, stats, warmup, repeat, flops, nbytes, machine):
    global _order
    _order += 1
    seconds = stats["mean_ms"] / 1000.0
    return {
        "exp": exp,
        "impl": impl,
        "device": device.type,
        "threads": threads,
        "size": size,
        **{k: round(v, 6) for k, v in stats.items()},
        "warmup": warmup,
        "repeats": repeat,
        "gflops": round(flops / seconds / 1e9, 3) if flops else "",
        "gbps": round(nbytes / seconds / 1e9, 3) if nbytes else "",
        "order": _order,
        "machine": machine,
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }


def append_rows(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    new_file = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=CSV_FIELDS)
        if new_file:
            writer.writeheader()
        writer.writerows(rows)


def cpu_name():
    system = platform.system()
    if system == "Windows":
        try:
            import winreg
            key = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE,
                                 r"HARDWARE\DESCRIPTION\System\CentralProcessor\0")
            return winreg.QueryValueEx(key, "ProcessorNameString")[0].strip()
        except OSError:
            pass
    elif system == "Linux":
        try:
            for line in Path("/proc/cpuinfo").read_text().splitlines():
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
        except OSError:
            pass
    return platform.processor() or "unknown"


def env_info(machine):
    info = {
        "machine": machine,
        "recorded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "os": platform.platform(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cpu": cpu_name(),
        "cpu_logical_cores": os.cpu_count(),
        "torch_default_threads": DEFAULT_THREADS,
        "mkl": torch.backends.mkl.is_available(),
        "mkldnn": torch.backends.mkldnn.is_available(),
        "cuda_available": torch.cuda.is_available(),
        "malloc_env": {k: v for k, v in os.environ.items() if k.startswith("MALLOC_")},
    }
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        info.update({
            "gpu": props.name,
            "gpu_count": torch.cuda.device_count(),
            "gpu_memory_gb": round(props.total_memory / 1024**3, 1),
            "gpu_sm_count": props.multi_processor_count,
            "gpu_compute_capability": f"{props.major}.{props.minor}",
            "cuda": torch.version.cuda,
            "cudnn": torch.backends.cudnn.version(),
        })
    return info


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")

In [ ]:
%%writefile /kaggle/working/code/gpu_sampler.py
"""Sample GPU 0 through NVML into a CSV until killed. Started by common.GpuClockLog.

    python gpu_sampler.py out.csv 20        # every 20 ms

A separate process, so it never competes with the benchmark for the Python GIL. Uses
ctypes on libnvidia-ml (installed with the driver), no extra package needed.
"""
import ctypes
import sys
import time


def check(code, what):
    if code != 0:
        raise RuntimeError(f"NVML {what} failed with code {code}")


def main():
    path, interval = sys.argv[1], float(sys.argv[2]) / 1000
    nvml = ctypes.CDLL("nvml.dll" if sys.platform == "win32" else "libnvidia-ml.so.1")
    check(nvml.nvmlInit_v2(), "init")
    handle = ctypes.c_void_p()
    check(nvml.nvmlDeviceGetHandleByIndex_v2(0, ctypes.byref(handle)), "handle")
    sm, mem, power, temp = (ctypes.c_uint() for _ in range(4))
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("t,sm_mhz,mem_mhz,power_w,temp_c\n")
        while True:
            check(nvml.nvmlDeviceGetClockInfo(handle, 1, ctypes.byref(sm)), "SM clock")
            check(nvml.nvmlDeviceGetClockInfo(handle, 2, ctypes.byref(mem)), "memory clock")
            # Power and temperature are extra information: 0 if the driver refuses them.
            nvml.nvmlDeviceGetPowerUsage(handle, ctypes.byref(power))
            nvml.nvmlDeviceGetTemperature(handle, 0, ctypes.byref(temp))
            fh.write(f"{time.time():.4f},{sm.value},{mem.value},{power.value / 1000:.1f},{temp.value}\n")
            fh.flush()
            time.sleep(interval)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/bench_conv.py
"""Experiment 1 (paper Fig 1-2): forward time of the paper's first conv layer vs batch size.

Layer from the paper: greyscale 224x224 input, 96 filters of 11x11, stride 4.
Output is 96 x 54 x 54, so each image unrolls (im2col) into a 121 x 2916 matrix.
(The paper says "54 parts"; it is 54 per side, i.e. 54*54 = 2916 patches.)

Implementations:
  caffe   - what the paper describes for Caffe: loop over the images of the batch one
            at a time; per image im2col (F.unfold) then one GEMM W(96x121) @ cols(121x2916)
            plus bias. This is the main series, comparable to the paper.
  batched - im2col for the whole batch, one batched GEMM (extension).
  conv2d  - the library convolution (oneDNN on CPU, cuDNN on GPU) (extension).

caffe and batched write into preallocated buffers (Caffe keeps its column buffer and output
blob too). F.conv2d has no out= argument, so on CPU it still allocates its output on every
call and the first-touch page faults of that buffer are part of its time. On Linux, run with
MALLOC_MMAP_MAX_=0 and a large MALLOC_TRIM_THRESHOLD_ so malloc reuses memory instead.
"""
import argparse

import torch
import torch.nn.functional as F

from common import (DEFAULT_THREADS, append_rows, bench, get_device, gpu_warmup, make_row,
                    ordered, parse_ints, resolve_threads)

IMG = 224
C_IN = 1
C_OUT = 96
K = 11
STRIDE = 4
OUT_HW = (IMG - K) // STRIDE + 1           # 54
PATCHES = OUT_HW * OUT_HW                   # 2916
PATCH_LEN = C_IN * K * K                    # 121
FLOPS_PER_IMAGE = 2 * C_OUT * PATCH_LEN * PATCHES   # multiply-adds of the GEMM

IMPLS = ("caffe", "batched", "conv2d")
DEFAULT_BATCHES = "1,8,16,32,64,96,128,160,192,224,256"


def make_layer(device):
    gen = torch.Generator().manual_seed(0)
    weight = torch.randn(C_OUT, C_IN, K, K, generator=gen).to(device)
    bias = torch.randn(C_OUT, generator=gen).to(device)
    return weight, bias


def im2col(x, out):
    return torch._C._nn.im2col(x, (K, K), (1, 1), (0, 0), (STRIDE, STRIDE), out=out)


def make_fn(impl, x, weight, bias):
    """Return a zero-argument callable that runs one forward pass of the layer."""
    w2 = weight.view(C_OUT, PATCH_LEN)        # unrolled kernels, 96 x 121
    b2 = bias.view(C_OUT, 1)
    n = x.shape[0]

    if impl == "caffe":
        out = torch.empty(n, C_OUT, PATCHES, device=x.device)       # Caffe's top blob
        cols = torch.empty(1, PATCH_LEN, PATCHES, device=x.device)  # Caffe's column buffer

        def fn():
            for i in range(n):
                im2col(x[i:i + 1], cols)                            # 121 x 2916
                torch.addmm(b2, w2, cols[0], out=out[i])            # 96 x 2916
            return out
    elif impl == "batched":
        out = torch.empty(n, C_OUT, PATCHES, device=x.device)
        cols = torch.empty(n, PATCH_LEN, PATCHES, device=x.device)

        def fn():
            im2col(x, cols)                                         # n x 121 x 2916
            return torch.matmul(w2, cols, out=out).add_(b2)         # n x 96 x 2916
    elif impl == "conv2d":
        def fn():
            return F.conv2d(x, weight, bias, stride=STRIDE)
    else:
        raise ValueError(f"unknown impl {impl}")
    return fn


def check_correctness(device):
    """All three implementations must give the same output before anything is timed."""
    weight, bias = make_layer(device)
    x = torch.randn(3, C_IN, IMG, IMG, generator=torch.Generator().manual_seed(1)).to(device)
    with torch.inference_mode():
        ref = make_fn("conv2d", x, weight, bias)().reshape(3, C_OUT, PATCHES)
        for impl in ("caffe", "batched"):
            got = make_fn(impl, x, weight, bias)().reshape(3, C_OUT, PATCHES)
            torch.testing.assert_close(got, ref, rtol=1e-3, atol=1e-2,
                                       msg=lambda m, impl=impl: f"{impl} != conv2d on {device}: {m}")


def run(device_name, threads_list, impls, batches, warmup, repeat, machine, out_csv, log=print,
        order_seed=0, warm_seconds=0):
    device = get_device(device_name)
    if device.type == "cuda":
        threads_list = [DEFAULT_THREADS]      # host threads do not matter for GPU runs
    check_correctness(device)
    weight, bias = make_layer(device)
    rows = []
    for threads in threads_list:
        torch.set_num_threads(threads)
        for impl in impls:
            gpu_warmup(device, warm_seconds)
            for n in ordered(batches, order_seed):
                x = torch.randn(n, C_IN, IMG, IMG, device=device)
                with torch.inference_mode():
                    fn = make_fn(impl, x, weight, bias)
                    stats = bench(fn, device, warmup, repeat)
                flops = FLOPS_PER_IMAGE * n
                nbytes = 4 * n * (IMG * IMG + C_OUT * PATCHES)   # read input + write output
                row = make_row("conv", impl, device, threads, n, stats, warmup, repeat,
                               flops, nbytes, machine)
                rows.append(row)
                log(f"conv  {device.type:4s} t={threads:<2d} {impl:7s} batch={n:<4d} "
                    f"{stats['mean_ms']:10.3f} ms ± {stats['std_ms']:.3f}  ({row['gflops']} GFLOP/s)"
                    + (f"  issue {stats['issue_ms']:.3f} ms" if device.type == "cuda" else ""))
                del x, fn
        append_rows(out_csv, [r for r in rows if r["threads"] == threads])
    torch.set_num_threads(DEFAULT_THREADS)
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return rows


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--device", default="cpu", choices=["cpu", "cuda"])
    parser.add_argument("--threads", default="1,max", help="CPU thread counts, e.g. 1,4,max")
    parser.add_argument("--impls", default=",".join(IMPLS))
    parser.add_argument("--batches", default=DEFAULT_BATCHES)
    parser.add_argument("--warmup", type=int, default=3)
    parser.add_argument("--repeat", type=int, default=10)
    parser.add_argument("--machine", default="local")
    parser.add_argument("--out", default="conv.csv")
    args = parser.parse_args()
    run(args.device, resolve_threads(args.threads), args.impls.split(","),
        parse_ints(args.batches), args.warmup, args.repeat, args.machine, args.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/bench_mem.py
"""Independent checks of the analysis (ANALYSIS.md section 7, PREDICTIONS.md), separate from the paper's runs.

    python bench_mem.py --machine laptop --quick                         # smoke test
    python bench_mem.py --machine kaggle --devices cuda,cpu --repeat 20  # one run

Three experiments; one CSV each in <outdir>/<machine>/<tag>, plus env.json, log.txt and gpu_clocks.csv:
  bw.csv    memory bandwidth from STREAM-like kernels that do not use matvec code:
              sum   read   4 B/element      dot   read 8 B/element
              copy  r+w    8 B/element      triad read 2 + write 1 = 12 B/element (STREAM counting)
            plus matvec itself. Arrays have N = n*n elements, matching the matvec sizes of fc.csv.
  fcb.csv   FC layer with B input vectors, (n x n) @ (n x B); impl mm1 (= mv) ... mm128.
  step.csv  Caffe-style conv timed as im2col only / GEMM only / both (the caffe impl), per batch.
The arrays are filled with random numbers so that no page is shared or compressed.
"""
import argparse
import time
from datetime import datetime
from pathlib import Path

import torch

import bench_conv
from common import (DEFAULT_THREADS, ROOT, GpuClockLog, append_rows, available_devices, bench, env_info,
                    get_device, gpu_warmup, make_row, ordered, parse_ints, resolve_threads, write_json)

BW_KERNELS = ("sum", "dot", "copy", "triad", "matvec")
DEFAULT_SIZES = "1024,4096,8192,16384"
DEFAULT_FC_SIZES = "2048,8192"
DEFAULT_B = "1,16,64,128"
DEFAULT_BATCHES = "16,64,256"
QUICK = dict(sizes="256,1024", fc_sizes="256,512", fc_b="1,8", batches="2,4", warmup=1, repeat=2)
TRIAD_S = 3.0


# ---------------------------------------------------------------- bandwidth

def bw_kernel(kernel, n, device):
    """Return (fn, flops, bytes) for one kernel over N = n*n float32 elements per array."""
    N = n * n
    gen = torch.Generator().manual_seed(n)
    x = torch.rand(N, generator=gen).to(device)
    if kernel == "sum":
        return (lambda: torch.sum(x)), N, 4 * N
    if kernel == "dot":
        y = torch.rand(N, generator=gen).to(device)
        return (lambda: torch.dot(x, y)), 2 * N, 8 * N
    if kernel == "copy":
        y = torch.empty(N, device=device)
        return (lambda: y.copy_(x)), 0, 8 * N
    if kernel == "triad":
        b = torch.rand(N, generator=gen).to(device)
        c, out = x, torch.empty(N, device=device)
        return (lambda: torch.add(b, c, alpha=TRIAD_S, out=out)), 2 * N, 12 * N
    if kernel == "matvec":
        a, v = x.view(n, n), torch.rand(n, generator=gen).to(device)
        out = torch.empty(n, device=device)
        return (lambda: torch.mv(a, v, out=out)), 2 * N, 4 * N
    raise ValueError(f"unknown kernel {kernel}")


def check_kernels(device):
    """The timed kernels must compute what they claim (small arrays, against NumPy-style references)."""
    n = 40
    for kernel in BW_KERNELS:
        fn, _, _ = bw_kernel(kernel, n, device)
        got = fn()
        x = torch.rand(n * n, generator=torch.Generator().manual_seed(n))
        if kernel == "sum":
            torch.testing.assert_close(got.cpu(), x.sum(), rtol=1e-4, atol=1e-3)
        elif kernel == "dot":
            # bw_kernel draws x first, then y from the same generator
            gen = torch.Generator().manual_seed(n)
            x2, y2 = torch.rand(n * n, generator=gen), torch.rand(n * n, generator=gen)
            torch.testing.assert_close(got.cpu(), (x2.double() @ y2.double()).float(), rtol=1e-4, atol=1e-3)
        elif kernel == "copy":
            torch.testing.assert_close(got.cpu(), x, rtol=0, atol=0)
        elif kernel == "triad":
            gen = torch.Generator().manual_seed(n)
            x2 = torch.rand(n * n, generator=gen)
            b2 = torch.rand(n * n, generator=gen)
            torch.testing.assert_close(got.cpu(), b2 + TRIAD_S * x2, rtol=1e-5, atol=1e-5)
        elif kernel == "matvec":
            gen = torch.Generator().manual_seed(n)
            x2 = torch.rand(n * n, generator=gen)
            v2 = torch.rand(n, generator=gen)
            torch.testing.assert_close(got.cpu(), (x2.view(n, n).double() @ v2.double()).float(), rtol=1e-4, atol=1e-3)


def run_bw(device_name, threads_list, sizes, warmup, repeat, machine, out_csv, log, order_seed, warm_seconds):
    device = get_device(device_name)
    if device.type == "cuda":
        threads_list = [DEFAULT_THREADS]
    check_kernels(device)
    rows = []
    for threads in threads_list:
        torch.set_num_threads(threads)
        for kernel in BW_KERNELS:
            gpu_warmup(device, warm_seconds)
            for n in ordered(sizes, order_seed):
                fn, flops, nbytes = bw_kernel(kernel, n, device)
                with torch.inference_mode():
                    stats = bench(fn, device, warmup, repeat)
                row = make_row("bw", kernel, device, threads, n, stats, warmup, repeat, flops, nbytes, machine)
                rows.append(row)
                log(f"bw    {device.type:4s} t={threads:<2d} {kernel:7s} n={n:<6d} {stats['mean_ms']:10.3f} ms ± "
                    f"{stats['std_ms']:.3f}  ({row['gbps']} GB/s)")
                del fn
                if device.type == "cuda":
                    torch.cuda.empty_cache()
        append_rows(out_csv, [r for r in rows if r["threads"] == threads])
    torch.set_num_threads(DEFAULT_THREADS)
    return rows


# ---------------------------------------------------------------- FC with B vectors

def fcb_kernel(n, B, device):
    gen = torch.Generator().manual_seed(n)
    a = torch.randn(n, n, generator=gen).to(device)
    if B == 1:
        x, y = torch.randn(n, generator=gen).to(device), torch.empty(n, device=device)
        return (lambda: torch.mv(a, x, out=y)), 2 * n * n, 4 * n * n
    x, y = torch.randn(n, B, generator=gen).to(device), torch.empty(n, B, device=device)
    return (lambda: torch.mm(a, x, out=y)), 2 * n * n * B, 4 * n * n


def check_fcb(device):
    fn, _, _ = fcb_kernel(200, 8, device)
    gen = torch.Generator().manual_seed(200)
    a = torch.randn(200, 200, generator=gen)
    x = torch.randn(200, 8, generator=gen)
    torch.testing.assert_close(fn().cpu(), (a.double() @ x.double()).float(), rtol=1e-4, atol=1e-3)


def run_fcb(device_name, threads_list, sizes, batches, warmup, repeat, machine, out_csv, log, order_seed, warm_seconds):
    device = get_device(device_name)
    if device.type == "cuda":
        threads_list = [DEFAULT_THREADS]
    check_fcb(device)
    rows = []
    for threads in threads_list:
        torch.set_num_threads(threads)
        gpu_warmup(device, warm_seconds)
        for n, B in ordered([(n, B) for n in sizes for B in batches], order_seed):
            fn, flops, nbytes = fcb_kernel(n, B, device)
            with torch.inference_mode():
                stats = bench(fn, device, warmup, repeat)
            row = make_row("fcb", f"mm{B}", device, threads, n, stats, warmup, repeat, flops, nbytes, machine)
            rows.append(row)
            log(f"fcb   {device.type:4s} t={threads:<2d} B={B:<4d} n={n:<6d} {stats['mean_ms']:10.3f} ms ± "
                f"{stats['std_ms']:.3f}  ({row['gflops']} GFLOP/s)")
            del fn
            if device.type == "cuda":
                torch.cuda.empty_cache()
        append_rows(out_csv, [r for r in rows if r["threads"] == threads])
    torch.set_num_threads(DEFAULT_THREADS)
    return rows


# ---------------------------------------------------------------- conv steps

def step_kernel(step, B, device):
    """im2col only, GEMM only, or both: the same per-image loop as bench_conv's 'caffe'."""
    weight, bias = bench_conv.make_layer(device)
    w2, b2 = weight.view(bench_conv.C_OUT, bench_conv.PATCH_LEN), bias.view(bench_conv.C_OUT, 1)
    x = torch.randn(B, bench_conv.C_IN, bench_conv.IMG, bench_conv.IMG, device=device)
    cols = torch.empty(1, bench_conv.PATCH_LEN, bench_conv.PATCHES, device=device)
    out = torch.empty(B, bench_conv.C_OUT, bench_conv.PATCHES, device=device)
    bench_conv.im2col(x[:1], cols)                        # GEMM-only needs valid, finite columns

    if step == "im2col":
        def fn():
            for i in range(B):
                bench_conv.im2col(x[i:i + 1], cols)
            return cols
        flops = 0
    elif step == "gemm":
        def fn():
            for i in range(B):
                torch.addmm(b2, w2, cols[0], out=out[i])
            return out
        flops = bench_conv.FLOPS_PER_IMAGE * B
    elif step == "caffe":
        fn = bench_conv.make_fn("caffe", x, weight, bias)
        flops = bench_conv.FLOPS_PER_IMAGE * B
    else:
        raise ValueError(step)
    return fn, flops


def run_steps(device_name, threads_list, batches, warmup, repeat, machine, out_csv, log, order_seed, warm_seconds):
    device = get_device(device_name)
    if device.type == "cuda":
        threads_list = [DEFAULT_THREADS]
    else:
        threads_list = [1]                                # the baseline of the paper comparison
    bench_conv.check_correctness(device)
    rows = []
    for threads in threads_list:
        torch.set_num_threads(threads)
        for step in ("im2col", "gemm", "caffe"):
            gpu_warmup(device, warm_seconds)
            for B in ordered(batches, order_seed):
                fn, flops = step_kernel(step, B, device)
                with torch.inference_mode():
                    stats = bench(fn, device, warmup, repeat)
                row = make_row("step", step, device, threads, B, stats, warmup, repeat, flops, 0, machine)
                rows.append(row)
                log(f"step  {device.type:4s} t={threads:<2d} {step:7s} batch={B:<4d} {stats['mean_ms']:10.3f} ms ± "
                    f"{stats['std_ms']:.3f}  ({stats['mean_ms'] / B * 1e3:.1f} us/img)")
                del fn
                if device.type == "cuda":
                    torch.cuda.empty_cache()
        append_rows(out_csv, [r for r in rows if r["threads"] == threads])
    torch.set_num_threads(DEFAULT_THREADS)
    return rows


# ---------------------------------------------------------------- main

def default_outdir():
    kaggle = Path("/kaggle/working")
    return kaggle / "results" if kaggle.is_dir() else ROOT / "results"


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--machine", required=True)
    p.add_argument("--tag", default=datetime.now().strftime("bw_%Y%m%d_%H%M%S"))
    p.add_argument("--outdir", type=Path, default=default_outdir())
    p.add_argument("--devices", default=",".join(available_devices()))
    p.add_argument("--threads", default="1,max")
    p.add_argument("--sizes", default=DEFAULT_SIZES, help="n for the bandwidth kernels (N = n*n elements)")
    p.add_argument("--fc-sizes", default=DEFAULT_FC_SIZES)
    p.add_argument("--fc-b", default=DEFAULT_B)
    p.add_argument("--batches", default=DEFAULT_BATCHES, help="for the conv steps")
    p.add_argument("--warmup", type=int, default=3)
    p.add_argument("--repeat", type=int, default=10)
    p.add_argument("--skip", default="", help="comma list: bw,fcb,step")
    p.add_argument("--order-seed", type=int, default=1)
    p.add_argument("--warm-seconds", type=float, default=5)
    p.add_argument("--quick", action="store_true")
    args = p.parse_args()
    if args.quick:
        args.sizes, args.fc_sizes, args.fc_b, args.batches = QUICK["sizes"], QUICK["fc_sizes"], QUICK["fc_b"], QUICK["batches"]
        args.warmup, args.repeat = QUICK["warmup"], QUICK["repeat"]
    run_dir = args.outdir / args.machine / args.tag
    run_dir.mkdir(parents=True, exist_ok=False)
    log_file = (run_dir / "log.txt").open("w", encoding="utf-8")

    def log(msg):
        print(msg, flush=True)
        log_file.write(msg + "\n")
        log_file.flush()

    env = env_info(args.machine)
    env["grid"] = {k: v for k, v in vars(args).items() if k not in ("outdir",)}
    write_json(run_dir / "env.json", env)
    log(f"run folder: {run_dir}")
    log(f"cpu: {env['cpu']} | torch threads (max): {env['torch_default_threads']}"
        + (f" | gpu: {env['gpu']}" if env["cuda_available"] else " | no CUDA GPU"))
    threads = resolve_threads(args.threads)
    skip = set(filter(None, args.skip.split(",")))
    sizes, fc_sizes = parse_ints(args.sizes), parse_ints(args.fc_sizes)
    fc_b, batches = parse_ints(args.fc_b), parse_ints(args.batches)
    start = time.perf_counter()
    for device in args.devices.split(","):
        with GpuClockLog(run_dir / "gpu_clocks.csv", device == "cuda", log):
            if "bw" not in skip:
                run_bw(device, threads, sizes, args.warmup, args.repeat, args.machine, run_dir / "bw.csv", log,
                       args.order_seed, args.warm_seconds)
            if "fcb" not in skip:
                run_fcb(device, threads, fc_sizes, fc_b, args.warmup, args.repeat, args.machine, run_dir / "fcb.csv", log,
                        args.order_seed, args.warm_seconds)
            if "step" not in skip:
                run_steps(device, threads, batches, args.warmup, args.repeat, args.machine, run_dir / "step.csv", log,
                          args.order_seed, args.warm_seconds)
    log(f"done in {time.perf_counter() - start:.1f} s -> {run_dir}")
    log_file.close()


if __name__ == "__main__":
    main()

## 3 · Helper: run a command, stream its output, stop the notebook if it fails

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
WORK = Path("/kaggle/working")
CODE, RES = WORK / "code", WORK / "results" / "kaggle"
ENV = dict(os.environ, MALLOC_MMAP_MAX_="0", MALLOC_TRIM_THRESHOLD_="68719476736", PYTHONUNBUFFERED="1")

def py(*args):
    cmd = [sys.executable, *map(str, args)]
    p = subprocess.Popen(cmd, cwd=CODE, env=ENV, text=True, encoding="utf-8",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"exit code {p.returncode}: {' '.join(cmd)}")

## 4 · Smoke test (a few seconds, kept out of the results)

In [ ]:
shutil.rmtree(WORK / "smoke", ignore_errors=True)
py("bench_mem.py", "--machine", "kaggle", "--quick", "--tag", "smoke", "--outdir", WORK / "smoke",
   "--devices", "cuda,cpu", "--warm-seconds", 1)

## 5 · 3 runs, GPU first, 10 repeats per point, shuffled sizes

In [ ]:
from datetime import datetime
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RES.mkdir(parents=True, exist_ok=True)
!lscpu > {RES}/lscpu_{STAMP}.txt

In [ ]:
RUN1 = RES / f"bw1_{STAMP}"
py("bench_mem.py", "--machine", "kaggle", "--tag", RUN1.name, "--outdir", RES.parent,
   "--devices", "cuda,cpu", "--repeat", 10, "--order-seed", 1)

In [ ]:
RUN2 = RES / f"bw2_{STAMP}"
py("bench_mem.py", "--machine", "kaggle", "--tag", RUN2.name, "--outdir", RES.parent,
   "--devices", "cuda,cpu", "--repeat", 10, "--order-seed", 2)

In [ ]:
RUN3 = RES / f"bw3_{STAMP}"
py("bench_mem.py", "--machine", "kaggle", "--tag", RUN3.name, "--outdir", RES.parent,
   "--devices", "cuda,cpu", "--repeat", 10, "--order-seed", 3)

## 6 · Quick look (means over the runs; the full analysis is done offline)

In [ ]:
import pandas as pd
bw = pd.concat([pd.read_csv(r / "bw.csv").assign(run=r.name) for r in (RUN1, RUN2, RUN3)])
print(bw.pivot_table(index=["device", "threads", "impl"], columns="size", values="gbps", aggfunc="mean").round(1).to_string())
fcb = pd.concat([pd.read_csv(r / "fcb.csv").assign(run=r.name) for r in (RUN1, RUN2, RUN3)])
print(fcb.pivot_table(index=["device", "threads", "impl"], columns="size", values="gflops", aggfunc="mean").round(1).to_string())
step = pd.concat([pd.read_csv(r / "step.csv").assign(run=r.name) for r in (RUN1, RUN2, RUN3)])
step["us_per_img"] = step["mean_ms"] / step["size"] * 1e3
print(step.pivot_table(index=["device", "impl"], columns="size", values="us_per_img", aggfunc="mean").round(1).to_string())

## 7 · Package the results for download

In [ ]:
print(shutil.make_archive(str(WORK / "results_bandwidth"), "zip", WORK, "results"))